In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

/Users/kartikvishnoi/.pyenv/versions/rag_pipeline/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [46]:
class AgentState(TypedDict):
    num1: int
    num2: int
    num3: int
    num4: int
    operation1: str
    operation2: str
    result1: int
    result2: int

In [47]:
def adder_node1(state: AgentState) -> AgentState:
    """Node to perform addition"""
    state['result'] = state['num1'] + state['num2']

    return state

def sub_node1(state: AgentState) -> AgentState:
    """Node to perform subtraction"""
    state['result1'] = state['num1'] - state['num2']

    return state

def adder_node2(state: AgentState) -> AgentState:
    """Node to perform addition"""
    state['result2'] = state['num3'] + state['num4']

    return state

def sub_node2(state: AgentState) -> AgentState:
    """Node to perform subtraction"""
    state['result2'] = state['num3'] - state['num4']

    return state

def first_router(state: AgentState):
    if state['operation1'] == '+':
        return "add1"
    else:
        return "sub1"

def second_router(state: AgentState):
    if state['operation2'] == '+':
        return "add2"
    else:
        return "sub2"


In [48]:
graph = StateGraph(AgentState)
graph.add_node('adder1', adder_node1)
graph.add_node('subber1', sub_node1)
graph.add_node('router1', lambda state:state)

graph.add_node('adder2', adder_node2)
graph.add_node('subber2', sub_node2)
graph.add_node('router2', lambda state:state)

graph.add_edge(START, 'router1')
graph.add_conditional_edges(
    "router1",
    first_router,
    {
        "add1": "adder1",
        "sub1": "subber1"
    }
)
graph.add_conditional_edges(
    'router2',
    second_router,
    {
        "add2": "adder2",
        "sub2": "subber2"
    }
)
graph.add_edge('adder1', 'router2')
graph.add_edge('subber1', 'router2')
graph.add_edge('adder2', END)
graph.add_edge('subber2', END)

app = graph.compile()

In [50]:
res = app.invoke({'num1': 10, 'num2': 2, 'num3': 15, 'num4': 10,
                  'operation1': '-', 'operation2':'+'})
res

{'num1': 10,
 'num2': 2,
 'num3': 15,
 'num4': 10,
 'operation1': '-',
 'operation2': '+',
 'result1': 8,
 'result2': 25}